In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
#df = pd.read_excel(file_path, sheet_name=sheet_name)
df0 = pd.read_excel("/content/sample_data/Book L8 data treatment and transform st.xlsx", "Customers")
print(f"shape {df0.shape} .... \n{df0.head()}")

shape (200, 14) .... 
    CustID  Gender      Race  BirthDate College  HouseholdSize  ZipCode  \
0  1530016  Female     Black 1986-12-16     Yes              5    90047   
1  1531136    Male     White 1993-05-09     Yes              5    90026   
2  1532160    Male     Black 1966-05-22     Yes              2    90027   
3  1532307    Male     White 1964-09-16     Yes              4    90029   
4  1532356  Female  Hispanic 1964-07-15      No              5    90017   

   Income  Spending2020  Spending2021  NumOfOrders  DaysSinceLast  \
0   53000           287           241            3            101   
1   94000          1227           843           12            262   
2   64000           523           719            9            122   
3   60000           516           582           13            129   
4   47000           555           845            7             97   

        Satisfaction Channel  
0  Very Dissatisfied      SM  
1            Neutral      TV  
2     Very Satisfie

In [3]:
df = df0.copy() # To reserve no change in the orgininal versionf of file

In [4]:
# prompt: Multiply -1 to DaysSinceLast for making Recency column, then transform Recency values into 5 bins (based on 20% percentile incrementally). Also make bins for the NumOfOrders to reflect Frequency, and Spending2021 to reflect the Monetary values. Create new columns to store the tranformed values.

# Calculate Recency
df['Recency'] = -1 * df['DaysSinceLast']

# Calculate quantiles for Recency
r_labels = range(1,6)
r_quantiles = pd.qcut(df['Recency'], q=5, labels = r_labels)
df['R_Bin'] = r_quantiles


# Calculate quantiles for NumOfOrders (Frequency)
f_labels = range(1,6)
f_quantiles = pd.qcut(df['NumOfOrders'], q=5, labels = f_labels)
df['F_Bin'] = f_quantiles


# Calculate quantiles for Spending2021 (Monetary)
m_labels = range(1,6)
m_quantiles = pd.qcut(df['Spending2021'], q=5, labels = m_labels)
df['M_Bin'] = m_quantiles


In [5]:
df.head()

,CustID,Gender,Race,BirthDate,College,HouseholdSize,ZipCode,Income,Spending2020,Spending2021,NumOfOrders,DaysSinceLast,Satisfaction,Channel,Recency,R_Bin,F_Bin,M_Bin
0,1530016,Female,Black,1986-12-16,Yes,5,90047,53000,287,241,3,101,Very Dissatisfied,SM,-101,4,1,1
1,1531136,Male,White,1993-05-09,Yes,5,90026,94000,1227,843,12,262,Neutral,TV,-262,2,4,4
2,1532160,Male,Black,1966-05-22,Yes,2,90027,64000,523,719,9,122,Very Satisfied,TV,-122,4,3,3
3,1532307,Male,White,1964-09-16,Yes,4,90029,60000,516,582,13,129,Very Dissatisfied,SM,-129,4,4,3
4,1532356,Female,Hispanic,1964-07-15,No,5,90017,47000,555,845,7,97,Very Dissatisfied,Web,-97,4,2,4


In [6]:
# Concatenate R_Quartile, F_Quartile, and M_Quartile into a new column
df['RFM_Bin'] = df['R_Bin'].astype(str) + df['F_Bin'].astype(str) + df['M_Bin'].astype(str)

df.head()


,CustID,Gender,Race,BirthDate,College,HouseholdSize,ZipCode,Income,Spending2020,Spending2021,NumOfOrders,DaysSinceLast,Satisfaction,Channel,Recency,R_Bin,F_Bin,M_Bin,RFM_Bin
0,1530016,Female,Black,1986-12-16,Yes,5,90047,53000,287,241,3,101,Very Dissatisfied,SM,-101,4,1,1,411
1,1531136,Male,White,1993-05-09,Yes,5,90026,94000,1227,843,12,262,Neutral,TV,-262,2,4,4,244
2,1532160,Male,Black,1966-05-22,Yes,2,90027,64000,523,719,9,122,Very Satisfied,TV,-122,4,3,3,433
3,1532307,Male,White,1964-09-16,Yes,4,90029,60000,516,582,13,129,Very Dissatisfied,SM,-129,4,4,3,443
4,1532356,Female,Hispanic,1964-07-15,No,5,90017,47000,555,845,7,97,Very Dissatisfied,Web,-97,4,2,4,424


In [7]:
# prompt: bin Income to 5 bins (linear span of bin 1-5), not percentile.

min_income = df['Income'].min()
max_income = df['Income'].max()
bins = np.linspace(min_income, max_income, 6)  # 6 points define 5 bins
labels = range(1, 6)
df['Income_Bin'] = pd.cut(df['Income'], bins=bins, labels=labels, include_lowest=True)

df.head()

,CustID,Gender,Race,BirthDate,College,HouseholdSize,ZipCode,Income,Spending2020,Spending2021,NumOfOrders,DaysSinceLast,Satisfaction,Channel,Recency,R_Bin,F_Bin,M_Bin,RFM_Bin,Income_Bin
0,1530016,Female,Black,1986-12-16,Yes,5,90047,53000,287,241,3,101,Very Dissatisfied,SM,-101,4,1,1,411,1
1,1531136,Male,White,1993-05-09,Yes,5,90026,94000,1227,843,12,262,Neutral,TV,-262,2,4,4,244,3
2,1532160,Male,Black,1966-05-22,Yes,2,90027,64000,523,719,9,122,Very Satisfied,TV,-122,4,3,3,433,2
3,1532307,Male,White,1964-09-16,Yes,4,90029,60000,516,582,13,129,Very Dissatisfied,SM,-129,4,4,3,443,2
4,1532356,Female,Hispanic,1964-07-15,No,5,90017,47000,555,845,7,97,Very Dissatisfied,Web,-97,4,2,4,424,1


In [8]:
#Bin Spending2021 into a new column of 'Membership Tier' based on the value < 250 as 'Bronze', < 1000 as 'Silver' otherwise 'Gold'

# Create 'Membership Tier' column based on 'Spending2021'
df['Membership Tier'] = pd.cut(df['Spending2021'], bins=[0, 250, 1000, float('inf')], labels=['Bronze', 'Silver', 'Gold'], right=False)

df.head()


,CustID,Gender,Race,BirthDate,College,HouseholdSize,ZipCode,Income,Spending2020,Spending2021,...,DaysSinceLast,Satisfaction,Channel,Recency,R_Bin,F_Bin,M_Bin,RFM_Bin,Income_Bin,Membership Tier
0,1530016,Female,Black,1986-12-16,Yes,5,90047,53000,287,241,...,101,Very Dissatisfied,SM,-101,4,1,1,411,1,Bronze
1,1531136,Male,White,1993-05-09,Yes,5,90026,94000,1227,843,...,262,Neutral,TV,-262,2,4,4,244,3,Silver
2,1532160,Male,Black,1966-05-22,Yes,2,90027,64000,523,719,...,122,Very Satisfied,TV,-122,4,3,3,433,2,Silver
3,1532307,Male,White,1964-09-16,Yes,4,90029,60000,516,582,...,129,Very Dissatisfied,SM,-129,4,4,3,443,2,Silver
4,1532356,Female,Hispanic,1964-07-15,No,5,90017,47000,555,845,...,97,Very Dissatisfied,Web,-97,4,2,4,424,1,Silver
